<a href="https://colab.research.google.com/github/yansi64/EJERCICIOS/blob/EJERCICIOS_GOOGLE_COLAB/Practica03_F%C3%A1brica_Botellas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import simpy
import random

# Parámetros del sistema
TIEMPO_MOLDEO = 30        # segundos (media)
TIEMPO_ENFRIAMIENTO = 60  # segundos (constante)
TIEMPO_INSPECCION = 45    # segundos (media)
TAMANO_LOTE_EMPAQUE = 100 # botellas por lote
NUM_MAQUINAS_MOLDEO = 2
NUM_INSPECTORES = 1

# Proceso de producción de una botella
def producir_botella(env, id_botella, moldeo, inspeccion, empaque):
    # Moldeo
    with moldeo.request() as req:
        yield req
        tiempo = random.normalvariate(TIEMPO_MOLDEO, 5)
        yield env.timeout(tiempo)
        print(f"Botella {id_botella} moldeada en {env.now:.1f}")

    # Enfriamiento
    yield env.timeout(TIEMPO_ENFRIAMIENTO)
    print(f"Botella {id_botella} enfriada en {env.now:.1f}")

    # Inspección
    with inspeccion.request() as req:
        yield req
        tiempo = random.expovariate(1.0/TIEMPO_INSPECCION)
        yield env.timeout(tiempo)
        print(f"Botella {id_botella} inspeccionada en {env.now:.1f}")

    # Empaque (se acumulan hasta formar un lote)
    empaque.append(id_botella)
    if len(empaque) >= TAMANO_LOTE_EMPAQUE:
        print(f"Lote de {TAMANO_LOTE_EMPAQUE} botellas empacado en {env.now:.1f}")
        empaque.clear()

# Generador de llegadas de botellas
def llegada_botellas(env, moldeo, inspeccion, empaque):
    id_botella = 0
    while True:
        yield env.timeout(random.expovariate(1/20))  # llegada cada ~20 seg
        id_botella += 1
        env.process(producir_botella(env, id_botella, moldeo, inspeccion, empaque))

# Configuración del entorno
env = simpy.Environment()
moldeo = simpy.Resource(env, NUM_MAQUINAS_MOLDEO)
inspeccion = simpy.Resource(env, NUM_INSPECTORES)
empaque = []

# Iniciar simulación
env.process(llegada_botellas(env, moldeo, inspeccion, empaque))
env.run(until=500)  # simular 500 segundos

Botella 1 moldeada en 55.9
Botella 2 moldeada en 57.4
Botella 3 moldeada en 87.0
Botella 1 enfriada en 115.9
Botella 2 enfriada en 117.4
Botella 1 inspeccionada en 124.1
Botella 4 moldeada en 127.0
Botella 5 moldeada en 135.3
Botella 3 enfriada en 147.0
Botella 6 moldeada en 166.8
Botella 2 inspeccionada en 172.5
Botella 4 enfriada en 187.0
Botella 3 inspeccionada en 188.6
Botella 7 moldeada en 189.5
Botella 8 moldeada en 195.0
Botella 5 enfriada en 195.3
Botella 4 inspeccionada en 201.2
Botella 5 inspeccionada en 219.3
Botella 10 moldeada en 223.3
Botella 9 moldeada en 225.3
Botella 6 enfriada en 226.8
Botella 7 enfriada en 249.5
Botella 8 enfriada en 255.0
Botella 10 enfriada en 283.3
Botella 9 enfriada en 285.3
Botella 11 moldeada en 289.0
Botella 6 inspeccionada en 289.4
Botella 7 inspeccionada en 297.6
Botella 12 moldeada en 312.1
Botella 13 moldeada en 322.9
Botella 14 moldeada en 344.4
Botella 11 enfriada en 349.0
Botella 15 moldeada en 351.2
Botella 8 inspeccionada en 366.6
Bot